# Anomalous Financial Transaction Detection

본 대회의 과제는 금융 거래 데이터에서 **이상 거래를 탐지하는 기능**을 개선하고 활용도를 높이는 분류 AI모델을 개발하는 것입니다.

특히, 클래스 불균형 문제를 해결하기 위해 오픈소스 생성형 AI 모델을 활용하여 부족한 클래스의 데이터를 보완하고, 이를 통해 분류 모델의 성능을 향상시키는 것이 핵심 목표입니다.

이러한 접근을 통해 금융보안에 특화된 데이터 분석 및 활용 역량을 강화하여 전문 인력을 양성하고, 금융권의 AI 활용 어려움에 따른 해결 방안을 함께 모색하며 금융 산업의 AI 활용 활성화를 지원하는 것을 목표로 합니다.

# Import Library

In [2]:
from google.colab import files
uploaded = files.upload()

Saving train_kor.csv to train_kor.csv
Saving train.csv to train.csv
Saving test_kor.csv to test_kor.csv
Saving sample_submission.csv to sample_submission.csv


In [11]:
uploaded1 = files.upload()

Saving syn_submission.csv to syn_submission.csv


In [5]:
# 제출 파일 생성 관련
import os
import zipfile

# 데이터 처리 및 분석
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

# 머신러닝 전처리
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# 머신러닝 모델
import xgboost as xgb

# 합성 데이터 생성
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

# To ignore all warnings
import warnings
warnings.filterwarnings('ignore')

# 생성 🏭

# Load Data

In [14]:
train_all = pd.read_csv("/content/train_kor.csv") # 경로는 각자 지정
test_all = pd.read_csv("/content/test_kor.csv") # 경로는 각자 지정
all_synthetic_data = pd.read_csv('/content/syn_submission.csv')

In [7]:
train_all.head()

,샘플 식별자 번호,고객 출생년도,고객 성별,고객명,주민번호,고객 등록일자,고객 등급,3개월 이내 금융/공동인증서 발급 여부,3개월 이내 사설인증서 발급 여부,3개월 이내 보안카드 및 OTP 발급 여부,...,마지막 ATM 거래 일자,마지막 영업점 거래 일자,7일 거래내역 중 1천만원 이상 입금 여부,7일 거래내역 중 미거래 계좌 여부,수취계좌의 거래중지계좌 해당 여부,3시간 이내 해당 수취계좌에 이체 횟수,해당 수취계좌와 거래한 횟수,60세 이후 iOS 첫 사용자,사기 시나리오 (예측 목표),계좌의 거래 재개 일자
0,TRAIN_000000,1980,male,이상호,BJWQxd-WBASPLJ,2003-01-06 18:38:01,B,0,1,0,...,2003-01-22 23:38:48,2003-01-22 23:38:48,1,1,1,0,0,0,m,2003-01-22 23:38:48
1,TRAIN_000001,1964,male,박상철,kurCwX-odPUXEt,2003-01-07 16:40:44,C,0,1,0,...,2003-01-21 21:29:08,2003-01-31 00:19:46,0,1,0,0,0,0,m,2003-01-19 21:29:08
2,TRAIN_000002,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,0,...,2003-01-31 07:13:28,2003-01-31 07:13:28,0,0,1,1,1,0,m,2003-01-31 07:13:28
3,TRAIN_000003,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,...,2003-01-31 11:49:56,2003-01-31 07:13:28,1,1,0,0,0,0,m,2003-01-31 07:13:28
4,TRAIN_000004,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,...,2003-01-31 11:49:56,2003-01-31 07:13:28,1,0,0,1,1,0,m,2003-01-31 07:13:28


In [8]:
train = train_all.drop(columns="샘플 식별자 번호")

## 원본 데이터와 concat

In [27]:
origin_train = train_all.drop(columns="샘플 식별자 번호")
train_total = origin_train
train_total.shape

(120000, 63)

In [28]:
train_total

,고객 출생년도,고객 성별,고객명,주민번호,고객 등록일자,고객 등급,3개월 이내 금융/공동인증서 발급 여부,3개월 이내 사설인증서 발급 여부,3개월 이내 보안카드 및 OTP 발급 여부,3개월 이내 개인정보 수정 여부,...,마지막 ATM 거래 일자,마지막 영업점 거래 일자,7일 거래내역 중 1천만원 이상 입금 여부,7일 거래내역 중 미거래 계좌 여부,수취계좌의 거래중지계좌 해당 여부,3시간 이내 해당 수취계좌에 이체 횟수,해당 수취계좌와 거래한 횟수,60세 이후 iOS 첫 사용자,사기 시나리오 (예측 목표),계좌의 거래 재개 일자
0,1980,male,이상호,BJWQxd-WBASPLJ,2003-01-06 18:38:01,B,0,1,0,1,...,2003-01-22 23:38:48,2003-01-22 23:38:48,1,1,1,0,0,0,m,2003-01-22 23:38:48
1,1964,male,박상철,kurCwX-odPUXEt,2003-01-07 16:40:44,C,0,1,0,0,...,2003-01-21 21:29:08,2003-01-31 00:19:46,0,1,0,0,0,0,m,2003-01-19 21:29:08
2,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,0,0,...,2003-01-31 07:13:28,2003-01-31 07:13:28,0,0,1,1,1,0,m,2003-01-31 07:13:28
3,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,0,...,2003-01-31 11:49:56,2003-01-31 07:13:28,1,1,0,0,0,0,m,2003-01-31 07:13:28
4,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,0,...,2003-01-31 11:49:56,2003-01-31 07:13:28,1,0,0,1,1,0,m,2003-01-31 07:13:28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,1991,female,유아름,MBpRXq-QfPDZwF,2010-08-23 09:16:18,C,0,1,0,0,...,2051-11-06 08:03:53,2051-09-12 07:41:31,0,1,1,0,0,0,m,2054-12-05 11:51:18
119996,1991,female,장지은,PchfPH-CxuDqYG,2012-11-14 12:57:03,B,0,1,1,0,...,2053-12-28 18:59:17,2053-12-18 23:04:56,0,0,0,2,2,0,m,2055-12-31 07:07:51
119997,2004,male,윤지훈,lEIytT-xVrZZIO,2024-07-15 06:28:05,B,1,1,0,0,...,2056-08-06 07:44:50,2056-07-31 16:38:49,0,1,0,0,0,0,m,2056-07-25 11:44:22
119998,2004,female,김경희,JQKetZ-JOWiLRU,2024-05-12 00:31:19,A,1,1,1,1,...,2058-06-17 12:34:39,2058-06-06 09:25:05,0,1,0,0,0,0,m,2058-05-15 01:14:42


# Data Preprocessing 1 : Select x, y

In [29]:
train_x = train_total.drop(columns=['사기 시나리오 (예측 목표)'])
train_y = train_total['사기 시나리오 (예측 목표)']

test_x = test_all.drop(columns=['샘플 식별자 번호'])

In [30]:
## 파생변수 추가 코드
# 1. 최근 잔고 급감 여부 (초기 잔고의 70% 이상 하락)
train_x['Is_Sudden_Balance_Drop'] = (train_x['거래 후 잔액'] < train_x['거래 전 잔액'] * 0.3).astype(int)
test_x['Is_Sudden_Balance_Drop'] = (test_x['거래 후 잔액'] < test_x['거래 전 잔액'] * 0.3).astype(int)

# 2. 고액 이체 탐지 (거래금액 / 하루한도 비율)
train_x['Amount_Per_Limit_Ratio'] = train_x['이체 금액'] / (train_x['1일 거래 한도'] + 1)
test_x['Amount_Per_Limit_Ratio'] = test_x['이체 금액'] / (test_x['1일 거래 한도'] + 1)

# 5. (VPN , 루팅 , 로밍) 중 하나이면서, 고객 등급이 C인지 여부
train_x['B_Flag_GradeC_DeviceAnomaly'] = (
    (train_x['고객 등급'] == 'C') &
    ((train_x['모바일 로밍 여부'] == 1) | (train_x['탈옥 및 루팅 여부'] == 1) | (train_x['VPN 사용 여부'] == 1))).astype(int)

test_x['B_Flag_GradeC_DeviceAnomaly'] = (
    (test_x['고객 등급'] == 'C') &
    ((test_x['모바일 로밍 여부'] == 1) |(test_x['탈옥 및 루팅 여부'] == 1) | (test_x['VPN 사용 여부'] == 1))).astype(int)

# 6. 유휴 계좌의 갑작스러운 거래 시도
# train_x['Inactive_Account_Suddenly_Used'] = ((train_x['7일 거래내역 중 미거래 계좌 여부'] == 1) & (train_x['이체 금액'] > 0)).astype(int)
# test_x['Inactive_Account_Suddenly_Used'] = ((test_x['7일 거래내역 중 미거래 계좌 여부'] == 1) & (test_x['이체 금액'] > 0)).astype(int)

# # 7. 접속 실패 후 성공 거래 (3회 이상 실패 시 플래그)
# train_x['High_Connection_Failure_Flag'] = (train_x['거래 시스템 접속 실패 횟수'] >= 3).astype(int)
# test_x['High_Connection_Failure_Flag'] = (test_x['거래 시스템 접속 실패 횟수'] >= 3).astype(int)

# 8. 거래금액 / 최근 한달 최대 거래금액
train_x['Amount_vs_Monthly_Max'] = train_x['이체 금액'] / (train_x['1개월 거래내역 중 최대 이체(출금) 금액'] + 1)
test_x['Amount_vs_Monthly_Max'] = test_x['이체 금액'] / (test_x['1개월 거래내역 중 최대 이체(출금) 금액'] + 1)

# # 11. 동일 수취계좌로 반복 전송 여부 (같은 계좌로 3회 초과 송금)
# transfer_counts_train = train_x.groupby(['암호화된 계좌번호', '수취인 계좌번호']).size().rename('이체횟수')
# train_x = train_x.merge(transfer_counts_train, on=['암호화된 계좌번호', '수취인 계좌번호'], how='left')
# train_x['Same_Account_Repeated_Transfer'] = (train_x['이체횟수'] > 3).astype(int)

# transfer_counts_test = test_x.groupby(['암호화된 계좌번호', '수취인 계좌번호']).size().rename('이체횟수')
# test_x = test_x.merge(transfer_counts_test, on=['암호화된 계좌번호', '수취인 계좌번호'], how='left')
# test_x['Same_Account_Repeated_Transfer'] = (test_x['이체횟수'] > 3).astype(int)

# 12. 고객 나이가 60세 이상이면서, 대출 유형이 담보대출인지 여부
# 기준 연도 설정
current_year = 2024

train_x['고객 나이'] = current_year - train_x['고객 출생년도']
train_x['L_Is_Elderly_Secured_Loan'] = (
    (train_x['고객 나이'] >= 60) & (train_x['대출 신청 유형(a: 없음, b: 신용대출, c: 담보대출, d: 할부금융, e: 기타)'] == 'c')
).astype(int)

test_x['고객 나이'] = current_year - test_x['고객 출생년도']
test_x['L_Is_Elderly_Secured_Loan'] = (
    (test_x['고객 나이'] >= 60) & (test_x['대출 신청 유형(a: 없음, b: 신용대출, c: 담보대출, d: 할부금융, e: 기타)'] == 'c')
).astype(int)

# A. 거리
train_x['A_Is_Large_Distance'] = (train['직전 거래 발생지와의 거리 차이'] >= 300).astype(int)
test_x['A_Is_Large_Distance'] = (train['직전 거래 발생지와의 거리 차이'] >= 300).astype(int)

# C. 키로깅을 통한 정보 탈취 거래
train_x['C_Keylogging_Risk'] = ((train_x['거래 시스템 접속 실패 횟수'] >= 3) & (train_x['키로깅 여부'] == 1)).astype(int)
test_x['C_Keylogging_Risk'] = ((test_x['거래 시스템 접속 실패 횟수'] >= 3) & (test_x['키로깅 여부'] == 1)).astype(int)

# E. 보이스피싱 인출 시도
train_x['E_Vishing_Withdrawal_Risk'] = ((train_x['거래 채널'] == 'ATM') & (train_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1)).astype(int)
test_x['E_Vishing_Withdrawal_Risk'] = ((test_x['거래 채널'] == 'ATM') & (test_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1)).astype(int)

# F. 고액 거래
train_x['G_MoneyLaundering_Risk'] = ((train_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) & (train_x['30일 이내 계좌 정지 해제 여부'] == 1) & train_x['거래에 사용한 단말기 OS'] == 'Others').astype(int)
test_x['G_MoneyLaundering_Risk'] = ((test_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) & (test_x['30일 이내 계좌 정지 해제 여부'] == 1) & test_x['거래에 사용한 단말기 OS'] == 'Others').astype(int)

# G. 자금세탁 거래
train_x['G_MoneyLaundering_Risk'] = ((train_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) & (train_x['30일 이내 계좌 정지 해제 여부'] == 1)).astype(int)
test_x['G_MoneyLaundering_Risk'] = ((test_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) & (test_x['30일 이내 계좌 정지 해제 여부'] == 1)).astype(int)

# # H. 소액 분산 이체
# same_amount_counts_train = train_x.groupby(['암호화된 계좌번호', '이체 금액']).size().rename('동일금액이체횟수').reset_index()
# train_x = train_x.merge(same_amount_counts_train, on=['암호화된 계좌번호', '이체 금액'], how='left')
# train_x['H_Repeated_Same_Amount_Transfer'] = (train_x['동일금액이체횟수'] >= 3).astype(int)

# same_amount_counts_test = test_x.groupby(['암호화된 계좌번호', '이체 금액']).size().rename('동일금액이체횟수').reset_index()
# test_x = test_x.merge(same_amount_counts_test, on=['암호화된 계좌번호', '이체 금액'], how='left')
# test_x['H_Repeated_Same_Amount_Transfer'] = (test_x['동일금액이체횟수'] >= 3).astype(int)

# J. 대포통장 거래
train_x['J_Suspicious_Small_Repeated_Transfer'] = ((train_x['수취계좌의 거래중지계좌 해당 여부'] == 1) & (train_x['7일 거래내역 중 미거래 계좌 여부'] == 1)).astype(int)
test_x['J_Suspicious_Small_Repeated_Transfer'] = ((test_x['수취계좌의 거래중지계좌 해당 여부'] == 1) & (test_x['7일 거래내역 중 미거래 계좌 여부'] == 1)).astype(int)

# K. 부업 사기 및 공범 계좌 활용
train_x['K_Affiliate_Scam_Risk'] = ((train_x['3시간 이내 해당 수취계좌에 이체 횟수'] >= 2) & (train_x['해당 수취계좌와 거래한 횟수'] >= 2)).astype(int)
test_x['K_Affiliate_Scam_Risk'] = ((test_x['3시간 이내 해당 수취계좌에 이체 횟수'] >= 2) & (test_x['해당 수취계좌와 거래한 횟수'] >= 2)).astype(int)


# Data Preprocessing 2 : 범주형 변수 인코딩

In [31]:
le_subclass = LabelEncoder()
train_y_encoded = le_subclass.fit_transform(train_y)

# 변환된 레이블 확인
for i, label in enumerate(le_subclass.classes_):
    print(f"원래 레이블: {label}, 변환된 숫자: {i}")

원래 레이블: a, 변환된 숫자: 0
원래 레이블: b, 변환된 숫자: 1
원래 레이블: c, 변환된 숫자: 2
원래 레이블: d, 변환된 숫자: 3
원래 레이블: e, 변환된 숫자: 4
원래 레이블: f, 변환된 숫자: 5
원래 레이블: g, 변환된 숫자: 6
원래 레이블: h, 변환된 숫자: 7
원래 레이블: i, 변환된 숫자: 8
원래 레이블: j, 변환된 숫자: 9
원래 레이블: k, 변환된 숫자: 10
원래 레이블: l, 변환된 숫자: 11
원래 레이블: m, 변환된 숫자: 12


In [32]:
from sklearn.preprocessing import OrdinalEncoder
from pandas.api.types import is_object_dtype, is_categorical_dtype, is_datetime64_any_dtype

# ✅ Step 1. 시간 차이 수치형으로 변환
for df in [train_x, test_x]:
    if '직전 거래와의 시간 차이' in df.columns:
        df['직전 거래와의 시간 차이'] = pd.to_timedelta(df['직전 거래와의 시간 차이'], errors='coerce').dt.total_seconds()

# ✅ Step 2. 범주형 컬럼 선택 (datetime 완전 제외)
categorical_columns = [
    col for col in train_x.columns
    if (is_object_dtype(train_x[col]) or is_categorical_dtype(train_x[col]))
    and not is_datetime64_any_dtype(train_x[col])
    and not pd.api.types.is_datetime64_ns_dtype(train_x[col])
]

# ✅ Step 3. OrdinalEncoder 정의
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# ✅ Step 4. 학습 데이터 인코딩
train_x_encoded = train_x.copy()
train_x_encoded[categorical_columns] = ordinal_encoder.fit_transform(train_x[categorical_columns].astype(str))

# ✅ Step 5. 테스트 데이터 인코딩 (문자열로 통일)
test_x_encoded = test_x.copy()
test_x_encoded[categorical_columns] = ordinal_encoder.transform(test_x[categorical_columns].astype(str))

# ✅ Step 6. 컬럼 순서와 dtype 맞추기
feature_order = train_x_encoded.columns.tolist()
test_x_encoded = test_x_encoded[feature_order]

for col in feature_order:
    test_x_encoded[col] = test_x_encoded[col].astype(train_x_encoded[col].dtype)


# Model Define

In [33]:
def clean_column_names(df):
    df.columns = (
        df.columns
        .str.replace(r"[^\w\d_]+", "_", regex=True)  # 한글은 유지, 특수문자 제거
        .str.strip("_")  # 양쪽 밑줄 제거
    )
    return df


In [34]:
train_x_encoded = clean_column_names(train_x_encoded)
test_x_encoded.columns = train_x_encoded.columns  # 순서와 이름 통일


In [35]:
!nvidia-smi

Tue May 13 07:17:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [41]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    task_type='CPU',     # ✅ 핵심
    devices='0',         # GPU 번호
    random_seed=42,
    loss_function='MultiClass',
    verbose=100
)


In [42]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    tree_method='gpu_hist',   # ✅ 핵심
    predictor='gpu_predictor',
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)


In [43]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    device='gpu',        # ✅ 핵심
    random_state=42
)


In [44]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stack_model = StackingClassifier(
    estimators=[
        ('cat', cat_model),
        ('xgb', xgb_model),
        ('lgb', lgb_model)
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)


In [45]:
import time

start = time.time()
stack_model.fit(train_x_encoded, train_y_encoded)
print("총 학습 시간:", round(time.time() - start, 2), "초")
predictions = stack_model.predict(test_x_encoded)

# 🎯 예측 결과 라벨 복원
predictions_label = le_subclass.inverse_transform(predictions.ravel().astype(int))

총 학습 시간: 1502.63 초


# Submission

In [46]:
# 칼럼명이 영어 컬럼으로 일치해야 코드가 돌아감
train = pd.read_csv("/content/train.csv")
a = train['Time_difference']
train = train.drop(['ID','Time_difference'],axis = 1)
train['Time_difference'] = a
all_synthetic_data.columns = train.columns
all_synthetic_data.head()

,Customer_Birthyear,Customer_Gender,Customer_personal_identifier,Customer_identification_number,Customer_registration_datetime,Customer_credit_rating,Customer_flag_change_of_authentication_1,Customer_flag_change_of_authentication_2,Customer_flag_change_of_authentication_3,Customer_flag_change_of_authentication_4,...,Last_bank_branch_transaction_datetime,Flag_deposit_more_than_tenMillion,Unused_account_status,Recipient_account_suspend_status,Number_of_transaction_with_the_account,Transaction_history_with_the_account,First_time_iOS_by_vulnerable_user,Fraud_Type,Transaction_resumed_date,Time_difference
0,1950,female,김영호,dhITBu-DbhAPPn,2006-04-19 20:05:53,D,0,1,1,0,...,2004-07-10 20:11:30,0,1,1,0,0,0,m,2013-05-21 14:18:18,0 days 00:01:38
1,1950,male,이재현,VUdWiC-wXhKmwF,2003-03-12 21:28:24,A,1,1,0,1,...,2016-06-04 05:48:40,0,0,1,1,0,0,m,2010-06-21 12:38:37,0 days 00:01:38
2,1965,female,최유진,UWTuyh-pgXndzV,2005-08-31 16:22:09,A,1,1,1,1,...,2004-07-10 20:11:30,0,1,1,0,0,0,m,2026-10-01 04:10:30,0 days 00:01:38
3,1960,male,이민재,BDBAtF-ZmBUHYl,2005-04-04 23:53:57,B,1,1,0,1,...,2005-01-20 13:36:09,0,1,1,0,0,0,m,2036-07-16 09:09:00,0 days 00:01:38
4,1968,male,김영환,KpdklD-ymHOSLQ,2004-08-22 13:01:08,B,1,1,1,1,...,2018-02-05 01:00:41,0,1,1,0,3,0,m,2022-04-08 09:31:08,0 days 00:01:38


In [47]:
# 분류 예측 결과 제출 데이터프레임(DataFrame)
# 분류 예측 결과 데이터프레임 파일명을 반드시 clf_submission.csv 로 지정해야합니다.
clf_submission = pd.read_csv("/content/sample_submission.csv")
clf_submission["Fraud_Type"] = predictions_label
clf_submission.head()

,ID,Fraud_Type
0,TEST_000000,j
1,TEST_000001,m
2,TEST_000002,m
3,TEST_000003,m
4,TEST_000004,m


In [34]:
# 합성 데이터 생성 결과 제출 데이터프레임(DataFrame)
# 합성 데이터 생성 결과 데이터프레임 파일명을 반드시 syn_submission.csv 로 지정해야합니다.
all_synthetic_data.head()

,Customer_Birthyear,Customer_Gender,Customer_personal_identifier,Customer_identification_number,Customer_registration_datetime,Customer_credit_rating,Customer_flag_change_of_authentication_1,Customer_flag_change_of_authentication_2,Customer_flag_change_of_authentication_3,Customer_flag_change_of_authentication_4,...,Last_bank_branch_transaction_datetime,Flag_deposit_more_than_tenMillion,Unused_account_status,Recipient_account_suspend_status,Number_of_transaction_with_the_account,Transaction_history_with_the_account,First_time_iOS_by_vulnerable_user,Fraud_Type,Transaction_resumed_date,Time_difference
0,1971,female,김영호,dhITBu-DbhAPPn,2012-11-25 07:17:26,D,0,1,1,1,...,2012-09-17 10:55:21,0,1,0,0,0,0,m,2031-07-08 04:20:05,0 days 00:01:38
1,1984,male,이재현,VUdWiC-wXhKmwF,2004-09-13 20:48:35,A,1,1,0,1,...,2004-07-10 20:11:30,0,1,1,1,0,0,m,2030-06-03 14:14:54,0 days 00:01:38
2,2004,female,최유진,UWTuyh-pgXndzV,2009-08-02 02:42:00,A,1,1,1,1,...,2006-11-10 10:28:45,0,1,0,0,0,0,m,2014-05-20 21:31:48,0 days 00:01:38
3,2003,male,이민재,BDBAtF-ZmBUHYl,2007-06-10 05:20:10,B,1,1,0,1,...,2004-07-10 20:11:30,0,0,1,0,0,0,m,2039-02-06 11:20:56,0 days 00:01:38
4,1993,female,김영환,KpdklD-ymHOSLQ,2005-09-26 05:26:00,C,1,1,1,0,...,2004-07-10 20:11:30,0,0,0,2,1,0,m,2004-07-10 20:11:30,0 days 00:01:38


In [48]:
'''
(*) 저장 시 각 파일명을 반드시 확인해주세요.
    1. 분류 예측 결과 데이터프레임 파일명 = clf_submission.csv
    2. 합성 데이터 생성 결과 데이터프레임 파일명 = syn_submission.csv

(*) 제출 파일(zip) 내에 두 개의 데이터프레임이 각각 위의 파일명으로 반드시 존재해야합니다.
(*) 파일명을 일치시키지 않으면 채점이 불가능합니다.
'''

# 폴더 생성 및 작업 디렉토리 변경
os.makedirs('./submission', exist_ok=True)
os.chdir("./submission/")

# CSV 파일로 저장
clf_submission.to_csv('./clf_submission.csv', encoding='UTF-8-sig', index=False)
all_synthetic_data.to_csv('./syn_submission.csv', encoding='UTF-8-sig', index=False)

# ZIP 파일 생성 및 CSV 파일 추가
with zipfile.ZipFile("../baseline_submission.zip", 'w') as submission:
    submission.write('clf_submission.csv')
    submission.write('syn_submission.csv')

print('Done.')

Done.


In [ ]:
##################3

In [ ]:
def get_xgb_feature_importance(model, feature_names):
    importance = model.feature_importances_
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values(by='importance', ascending=False)
    return importance_df

In [ ]:
importance_df = get_xgb_feature_importance(model, train_x_encoded.columns)
print(importance_df.tail(10))  # 하위 30개만 보기

In [ ]:

def plot_xgb_feature_importance(model, feature_names, top_n=20):
    # ✅ 한글 폰트 설정 (Windows 환경 기준 기본 폰트 사용)
    plt.rcParams['font.family'] = 'AppleGothic'  # 윈도우: 맑은 고딕
    plt.rcParams['axes.unicode_minus'] = False     # 마이너스 깨짐 방지

    # 중요도 추출
    importance = model.feature_importances_
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values(by='importance', ascending=False)

    # 시각화
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df['feature'][:top_n][::-1], importance_df['importance'][:top_n][::-1])
    plt.xlabel('중요도')
    plt.title(f'Top {top_n} 중요 변수 (XGBoost)')
    plt.tight_layout()
    plt.show()

    return importance_df


In [ ]:
# 중요도 확인 + 시각화
importance_df = plot_xgb_feature_importance(model, train_x_encoded.columns, top_n=20)

In [ ]:
importance_df['feature'].tail(20)

In [49]:

train_x.columns
# 컬럼개수 확인
print("train_x columns:", train_x.columns)
print("train_x shape:", train_x.shape)

train_x columns: Index(['고객 출생년도', '고객 성별', '고객명', '주민번호', '고객 등록일자', '고객 등급',
       '3개월 이내 금융/공동인증서 발급 여부', '3개월 이내 사설인증서 발급 여부',
       '3개월 이내 보안카드 및 OTP 발급 여부', '3개월 이내 개인정보 수정 여부', '탈옥 및 루팅 여부',
       '모바일 로밍 여부', 'VPN 사용 여부',
       '대출 신청 유형(a: 없음, b: 신용대출, c: 담보대출, d: 할부금융, e: 기타)', '전화번호 조작 여부',
       '원격제어 여부', '템퍼링 여부', '피싱 여부', '신뢰할 수 없는 인증서 사용 여부', '키로깅 여부',
       '7일 이내 ATM 한도 문의 여부', '7일 이내 ATM 한도 증액 여부', '암호화된 계좌번호',
       '계좌유형(a: 입출금계좌, b: CMA, c: ISA, d: 저축계좌)', '계좌 개설 일자', '거래 전 잔액',
       '거래 후 잔액', '거래한도 증가 여부', '1일 거래 한도', '오픈뱅킹 사용 여부', '1일 거래 한도 잔여액',
       '30일 이내 계좌 정지 해제 여부', '1개월 거래내역 중 최대 이체(출금) 금액',
       '1개월 거래내역 이체(출금) 금액 표준편차(중앙값)', '1개월 새벽 거래내역 중 최대 이체(출금) 금액',
       '1개월 새벽 거래내역 이체(출금) 금액 표준편처(중앙값)', '거래일자', '이체 금액', '거래 채널',
       '거래에 사용한 단말기 OS',
       '에러코드(a: 에러없음, b: 시스템 오류, c: 잔액부족, d: 이체한도초과, e: 계좌정보 오류, f: 계좌이체 거부)',
       '거래 성공/실패 여부', '자동이체(automatic) 또는 일반거래(general)', '거래에 사용한 단말기 IP주소',
       '거래에 사용한 단말기 MAC주소',
       '